# Chantier 1 — CatBoost + Stacking Ridge

Etape 1 : CatBoost solo (Optuna 30 essais — reduit par contrainte de session, justification dans le rapport)
Etape 2 : Stacking Ridge sur 4 modeles base (XGB v2, LGBM quantile P50, CatBoost, XGB-Tweedie si dispo)

Cible : MAE stacking < MAE XGB v2 (8.42).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib
import optuna
from catboost import CatBoostRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from dashboard.utils.features import FEATURES_V2, TARGET_QTE

DATA = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'
REPORTS = ROOT / 'reports'
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
df_train = pd.read_parquet(DATA / 'split_train_v3_features.parquet')
df_val = pd.read_parquet(DATA / 'split_val_v3_features.parquet')
df_test = pd.read_parquet(DATA / 'split_test_v3_features.parquet')

X_train = df_train[FEATURES_V2]; y_train_log = np.log1p(df_train[TARGET_QTE].clip(lower=0))
X_val = df_val[FEATURES_V2]; y_val_log = np.log1p(df_val[TARGET_QTE].clip(lower=0))
X_test = df_test[FEATURES_V2]; y_test = df_test[TARGET_QTE].values
print('Shapes:', X_train.shape, X_val.shape, X_test.shape)

Shapes: (210641, 47) (70871, 47) (66174, 47)


## 1. Optuna CatBoost (30 essais)

In [3]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 900),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'loss_function': 'RMSE', 'eval_metric': 'MAE', 'verbose': False,
        'allow_writing_files': False, 'random_state': 42,
    }
    m = CatBoostRegressor(**params)
    m.fit(X_train, y_train_log, eval_set=(X_val, y_val_log), early_stopping_rounds=30, verbose=False)
    pred = np.clip(np.expm1(m.predict(X_val)), 0, None)
    return mean_absolute_error(np.expm1(y_val_log), pred)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=False)
print('Best MAE val:', study.best_value)
print('Best params:', study.best_params)

Best MAE val: 8.611330311035728
Best params: {'iterations': 705, 'depth': 9, 'learning_rate': 0.034480628100521264, 'l2_leaf_reg': 0.04424137692640375, 'bagging_temperature': 0.6896744252117673, 'random_strength': 0.033168887516857926, 'min_data_in_leaf': 74}


In [4]:
best_params = {**study.best_params, 'loss_function': 'RMSE', 'eval_metric': 'MAE',
               'verbose': False, 'allow_writing_files': False, 'random_state': 42}
cat_model = CatBoostRegressor(**best_params)
cat_model.fit(X_train, y_train_log, eval_set=(X_val, y_val_log), early_stopping_rounds=30, verbose=False)
joblib.dump(cat_model, MODELS / 'catboost_v2.pkl')
with open(MODELS / 'catboost_v2_params.json', 'w', encoding='utf-8') as f:
    json.dump({'best_val_mae': study.best_value, 'best_params': study.best_params, 'n_trials': 30}, f, indent=2)

pred_cat_test = np.clip(np.expm1(cat_model.predict(X_test)), 0, None)
mae_cat = mean_absolute_error(y_test, pred_cat_test)
rmse_cat = np.sqrt(mean_squared_error(y_test, pred_cat_test))
r2_cat = r2_score(y_test, pred_cat_test)
print(f'CatBoost test MAE={mae_cat:.3f} RMSE={rmse_cat:.2f} R2={r2_cat:.3f}')

CatBoost test MAE=8.980 RMSE=120.05 R2=0.536


## 2. Stacking Ridge (4 modeles base)

In [5]:
# Predictions des modeles base sur val et test (espace log puis retour normal)
xgb_v2 = joblib.load(MODELS / 'xgboost_optuna_v2.pkl')
lgbm_p50 = joblib.load(MODELS / 'lgbm_quantile_p50.pkl')

preds_val = {
    'xgb_v2': np.clip(np.expm1(xgb_v2.predict(X_val)), 0, None),
    'lgbm_p50': np.clip(np.expm1(lgbm_p50.predict(X_val)), 0, None),
    'catboost': np.clip(np.expm1(cat_model.predict(X_val)), 0, None),
}
preds_test = {
    'xgb_v2': np.clip(np.expm1(xgb_v2.predict(X_test)), 0, None),
    'lgbm_p50': np.clip(np.expm1(lgbm_p50.predict(X_test)), 0, None),
    'catboost': np.clip(np.expm1(cat_model.predict(X_test)), 0, None),
}

# Tweedie XGB si dispo
tweedie_path = MODELS / 'xgboost_tweedie_qte.pkl'
if tweedie_path.exists():
    try:
        twd = joblib.load(tweedie_path)
        # Le modele Tweedie ancien utilisait FEATURES v1 (28). On verifie.
        n_features_twd = getattr(twd, 'n_features_in_', None)
        if n_features_twd == len(FEATURES_V2):
            preds_val['xgb_tweedie'] = np.clip(twd.predict(X_val), 0, None)
            preds_test['xgb_tweedie'] = np.clip(twd.predict(X_test), 0, None)
        else:
            print(f'Tweedie XGB ignore (features {n_features_twd} != V2 {len(FEATURES_V2)})')
    except Exception as e:
        print('Tweedie XGB ignore :', e)

stacking_components = list(preds_val.keys())
print('Composants stacking :', stacking_components)

P_val = np.column_stack([preds_val[k] for k in stacking_components])
P_test = np.column_stack([preds_test[k] for k in stacking_components])
y_val = np.expm1(y_val_log).values

ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0])
ridge.fit(P_val, y_val)
pred_stack_test = np.clip(ridge.predict(P_test), 0, None)
joblib.dump(ridge, MODELS / 'stacking_ridge.pkl')
with open(MODELS / 'stacking_components.json', 'w', encoding='utf-8') as f:
    json.dump({'components': stacking_components, 'best_alpha': float(ridge.alpha_),
               'coef': ridge.coef_.tolist(), 'intercept': float(ridge.intercept_)}, f, indent=2)

mae_stack = mean_absolute_error(y_test, pred_stack_test)
rmse_stack = np.sqrt(mean_squared_error(y_test, pred_stack_test))
r2_stack = r2_score(y_test, pred_stack_test)
print(f'Stacking test MAE={mae_stack:.3f} RMSE={rmse_stack:.2f} R2={r2_stack:.3f}')

Tweedie XGB ignore (features 28 != V2 47)
Composants stacking : ['xgb_v2', 'lgbm_p50', 'catboost']
Stacking test MAE=8.702 RMSE=97.55 R2=0.694


In [6]:
# Tableau comparatif consolide
results = []
for name, pred in [('XGBoost v2', preds_test['xgb_v2']), ('LGBM P50', preds_test['lgbm_p50']),
                   ('CatBoost', pred_cat_test), ('Stacking Ridge', pred_stack_test)]:
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    wape = np.sum(np.abs(y_test - pred)) / np.sum(np.abs(y_test))
    results.append({'modele': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'WAPE': wape})
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

with open(REPORTS / 'sprint_b_chantier1_stacking.json', 'w', encoding='utf-8') as f:
    json.dump({
        'results': df_results.to_dict(orient='records'),
        'best_alpha_ridge': float(ridge.alpha_),
        'stacking_components': stacking_components,
        'critere_succes_stacking_battu_xgb': bool(mae_stack < min(mean_absolute_error(y_test, preds_test['xgb_v2']),
                                                                  mean_absolute_error(y_test, preds_test['lgbm_p50']),
                                                                  mae_cat)),
    }, f, indent=2, ensure_ascii=False)

        modele      MAE       RMSE       R2     WAPE
    XGBoost v2 8.422108 101.834237 0.666347 0.383424
      LGBM P50 8.322904 121.871203 0.522130 0.378908
      CatBoost 8.980088 120.047202 0.536327 0.408826
Stacking Ridge 8.702106  97.551969 0.693818 0.396171


## Conclusion

Le stacking est valide si MAE Stacking < min(MAE des bases). Sinon, on garde XGBoost v2 et on justifie la decision dans le rapport (Occam : preferer le modele simple si gain marginal).